In [2]:
import pandas as pd

# WEATHER
weather = pd.read_csv(
    "../datasets/weather/up_weather_2015_2024.csv",
    skiprows=11
)

# YIELD
yield_data = pd.read_csv(
    "../datasets/yield/crop_production.csv"
)

# NDVI
ndvi = pd.read_csv(
    "../datasets/satellite/UP_Yearly_NDVI_2015_2024.csv"
)

print(weather.head())
print(yield_data.head())
print(ndvi.head())

   YEAR  DOY    T2M   RH2M  PRECTOTCORR
0  2015    1  15.55  50.47         0.28
1  2015    2  17.12  73.35        19.02
2  2015    3  18.07  82.78         5.55
3  2015    4  17.08  67.61         0.29
4  2015    5  14.10  47.95         0.03
                    State_Name District_Name  Crop_Year       Season  \
0  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
1  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
2  Andaman and Nicobar Islands      NICOBARS       2000  Kharif        
3  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    
4  Andaman and Nicobar Islands      NICOBARS       2000  Whole Year    

                  Crop    Area  Production  
0             Arecanut  1254.0      2000.0  
1  Other Kharif pulses     2.0         1.0  
2                 Rice   102.0       321.0  
3               Banana   176.0       641.0  
4            Cashewnut   720.0       165.0  
       NDVI    Year                                    .g

In [3]:
# Create yearly averages

yearly_weather = weather.groupby('YEAR').agg({
    'T2M': 'mean',
    'RH2M': 'mean',
    'PRECTOTCORR': 'sum'
}).reset_index()

# Rename YEAR column
yearly_weather.rename(
    columns={'YEAR': 'Year'},
    inplace=True
)

print(yearly_weather.head())

   Year        T2M       RH2M  PRECTOTCORR
0  2015  26.546466  46.651151       737.89
1  2016  26.391230  49.130410      1050.69
2  2017  26.231397  51.598548      1128.51
3  2018  25.947151  52.359288      1381.52
4  2019  25.403644  58.926575      1300.60


In [4]:
# Keep only required columns

ndvi = ndvi[['Year', 'NDVI']]

print(ndvi.head())

     Year      NDVI
0  2015.0  0.399717
1  2016.0  0.382146
2  2017.0  0.376525
3  2018.0  0.403998
4  2019.0  0.376034


In [5]:
# Filter Uttar Pradesh wheat data

yield_filtered = yield_data[
    (yield_data['State_Name'] == 'Uttar Pradesh') &
    (yield_data['Crop'] == 'Wheat') &
    (yield_data['Crop_Year'] >= 2015) &
    (yield_data['Crop_Year'] <= 2024)
]

print(yield_filtered.head())


yield_filtered['Yield'] = (
    yield_filtered['Production'] /
    yield_filtered['Area']
)

Empty DataFrame
Columns: [State_Name, District_Name, Crop_Year, Season, Crop, Area, Production]
Index: []


In [6]:
yearly_yield = yield_filtered.groupby(
    'Crop_Year'
)['Yield'].mean().reset_index()

# Rename column
yearly_yield.rename(
    columns={'Crop_Year': 'Year'},
    inplace=True
)

print(yearly_yield)

Empty DataFrame
Columns: [Year, Yield]
Index: []


In [7]:
merged = pd.merge(
    yearly_weather,
    ndvi,
    on='Year'
)

print(merged.head())

   Year        T2M       RH2M  PRECTOTCORR      NDVI
0  2015  26.546466  46.651151       737.89  0.399717
1  2016  26.391230  49.130410      1050.69  0.382146
2  2017  26.231397  51.598548      1128.51  0.376525
3  2018  25.947151  52.359288      1381.52  0.403998
4  2019  25.403644  58.926575      1300.60  0.376034


In [8]:
final_df = pd.merge(
    merged,
    yearly_yield,
    on='Year'
)

print(final_df.head())

Empty DataFrame
Columns: [Year, T2M, RH2M, PRECTOTCORR, NDVI, Yield]
Index: []


In [9]:
# Convert all Year columns to integer

yearly_weather['Year'] = yearly_weather['Year'].astype(int)

ndvi['Year'] = ndvi['Year'].astype(int)

yearly_yield['Year'] = yearly_yield['Year'].astype(int)

In [10]:
print(yearly_weather['Year'])
print(ndvi['Year'])
print(yearly_yield['Year'])

0    2015
1    2016
2    2017
3    2018
4    2019
5    2020
6    2021
7    2022
8    2023
9    2024
Name: Year, dtype: int64
0    2015
1    2016
2    2017
3    2018
4    2019
5    2020
6    2021
7    2022
8    2023
9    2024
Name: Year, dtype: int64
Series([], Name: Year, dtype: int64)


In [11]:
merged = pd.merge(
    yearly_weather,
    ndvi,
    on='Year'
)

final_df = pd.merge(
    merged,
    yearly_yield,
    on='Year'
)

print(final_df)

Empty DataFrame
Columns: [Year, T2M, RH2M, PRECTOTCORR, NDVI, Yield]
Index: []


In [12]:
final_df.to_csv(
    "../datasets/final_dataset.csv",
    index=False
)

In [13]:
print(yearly_weather.head())
print(ndvi.head())
print(yearly_yield.head())

print(yearly_weather.dtypes)
print(ndvi.dtypes)
print(yearly_yield.dtypes)

   Year        T2M       RH2M  PRECTOTCORR
0  2015  26.546466  46.651151       737.89
1  2016  26.391230  49.130410      1050.69
2  2017  26.231397  51.598548      1128.51
3  2018  25.947151  52.359288      1381.52
4  2019  25.403644  58.926575      1300.60
   Year      NDVI
0  2015  0.399717
1  2016  0.382146
2  2017  0.376525
3  2018  0.403998
4  2019  0.376034
Empty DataFrame
Columns: [Year, Yield]
Index: []
Year             int64
T2M            float64
RH2M           float64
PRECTOTCORR    float64
dtype: object
Year      int64
NDVI    float64
dtype: object
Year       int64
Yield    float64
dtype: object
